# Coffee Law ceilings: small-scale run

This notebook runs the ceiling-discrimination experiment suite
(`specs/001-ceiling-discrimination-experiments/`) at small scale and plots the results.

**What it runs by default:** the `sim-bounded` simulated agent. It is not a language model. Its
ceilings are known by construction (an explicit step budget per reasoning-depth level, exact
Bayesian updating on probe results, and messages limited to what the channel carries). So these
results **validate the harness and the analysis** (does the pipeline recover a known answer?).
They are **not evidence for or against the Coffee Law.** No API key is needed and nothing is billed.

**Optional:** the last section runs a tiny Claude pilot. It is off by default and capped by
`MAX_COST_USD`.

**Order:** setup → theory checks → calibration gate and cost matching → main grid (9 arms × 3
delta levels × 4 families) → gain table and verdicts H1–H5 → Fano sweep (H6) → communication
threshold (H7) → usable vs available vs budget → audit → save outputs.

## 0. Setup on GCP

This project lives in `experiments/ceilings/` of the private repository `Ishan-K-S/coffeelaw`. Any
of these works (Python 3.10 or newer):

1. **Vertex AI Workbench / Colab Enterprise / a VM with Jupyter.** Clone the repository (it is
   private, so authenticate first, e.g. `gh auth login` or a deploy key):
   `git clone https://github.com/Ishan-K-S/coffeelaw.git && cd coffeelaw/experiments/ceilings`.
   Then open `notebooks/ceilings_small_run.ipynb`. The next cell installs the package.
2. **Docker on a VM.** From `experiments/ceilings/`: `docker build -t ceilings .`, then
   `docker run -p 8888:8888 -v "$PWD":/work ceilings jupyter notebook --ip 0.0.0.0 --allow-root --no-browser`.

The default (`SCALE = "small"`) needs no GPU and runs in a few minutes on a small machine.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

assert sys.version_info >= (3, 10), f"needs Python 3.10+, found {sys.version}"

# Find the repository root (the folder that holds pyproject.toml), whether the notebook was
# opened from notebooks/ or from the root.
here = Path.cwd()
ROOT = next(p for p in [here, *here.parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)

try:
    import ceilings  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[notebook]"])
    sys.path.insert(0, str(ROOT / "src"))  # usable without a kernel restart

import ceilings

print("repo root:", ROOT)
print("ceilings", ceilings.__version__, "| Python", sys.version.split()[0])

## 1. Configuration

In [ ]:
SCALE = "small"            # "small" (minutes) or "medium" (closer to the planned pilot)
MODEL = "sim-bounded"      # the simulated agent; see section 9 for an optional Claude pilot
OUT = ROOT / "nb_data"     # all instances, runs, transcripts and reports go here
N_BOOT = 2000              # bootstrap resamples for confidence intervals

SIZES = {
    "small":  {"n_main": 200, "n_pilot": 200, "n_sweep": 150},
    "medium": {"n_main": 850, "n_pilot": 200, "n_sweep": 400},
}[SCALE]
FAMILIES = ["info", "reason", "comm_parity", "comm_ip"]
DELTAS = ("small", "medium", "large")
SWEEP_K, SWEEP_T, SWEEP_P = 4, [0, 1, 2, 3, 4, 6, 8, 10, 12, 16], [0.0, 0.1, 0.2]
SWEEP_NBITS, SWEEP_L = 16, [0, 1, 2, 4, 6, 8, 10, 12, 14, 15, 16]
OUT.mkdir(exist_ok=True)
print(SCALE, SIZES)

## 2. Theory checks

These are the reference numbers from the literature review (`litreview-coffee-law-check.py`). If
any assertion fails, stop: the theory module is wrong and nothing downstream can be trusted.

In [ ]:
from ceilings import theory

C = theory.bsc_capacity(0.1)
checks = {
    "capacity of a 10%-noisy yes/no probe (bits)": (C, 0.531),
    "budget after two probes (bits)": (theory.budget(0, [C, C]), 1.062),
    "information, bit 0 then bit 1": (theory.probe_information(2, 0.1, 2), 1.062),
    "MAP error after two probes": (theory.probe_map_error(2, 0.1, 2), 0.19),
    "Corollary 2, k=10 flaky test, eps=0.1 (rounds)": (theory.cor2_min_rounds(10, C, 0.1), 15.066),
    "IP n=4, best one-bit message (bits)": (theory.best_one_bit_message_ip(4), 0.1222),
    "IP bound n=100, L=45 (bits)": (theory.ip_bound(100, 45), 0.0875),
}
for name, (got, want) in checks.items():
    assert abs(got - want) < 1e-3, (name, got, want)
    print(f"{name:52s} {got:8.4f}  (expected {want})")

## 3. Calibration gate and cost matching

For each family:

- **Calibration gate (FR-007):** baseline accuracy must sit at chance (within 5 points, allowing
  for sampling error), and the family's own intervention at full strength must reach ≥ 90%. This
  establishes which ceiling binds, rather than assuming it.
- **Cost matching (FR-009):** every arm's added cost must land within ±5% of each target delta
  (0.5×, 1× and 2× the baseline cost, in tokens). Arms whose natural cost is lower are topped up
  with neutral filler text; arm P2 measures what filler alone does.

In [ ]:
import time

import pandas as pd

from ceilings import experiment
from ceilings.agents import make_policy

policy = make_policy(MODEL)
prepared = {}
t0 = time.time()
for fam in FAMILIES:
    prepared[fam] = experiment.prepare(fam, policy, OUT, SIZES["n_main"], SIZES["n_pilot"])
print(f"prepared {len(FAMILIES)} families in {time.time() - t0:.1f}s")

gate = pd.DataFrame([prepared[f]["calibration"].model_dump() for f in FAMILIES])
gate[["family_id", "n", "baseline_acc", "chance_acc", "matched_full_acc", "leak_rate", "passed"]]

In [ ]:
rows = []
for fam in FAMILIES:
    table = prepared[fam]["table"]
    for arm, by_level in table["arms"].items():
        for lvl, e in by_level.items():
            rows.append({"family": fam, "arm": arm, "delta": lvl, "strength": e["strength"],
                         "filler": e["filler_tokens"], "target": e["target"],
                         "added": round(e["added"]), "matched": e["matched"]})
match = pd.DataFrame(rows)
print("cells within ±5% of target:", f"{match['matched'].mean():.0%}")
match.pivot_table(index=["family", "arm"], columns="delta", values="matched", aggfunc="first")

## 4. Main grid

Every main-split instance runs under baseline plus the 8 other arms at each of 3 delta levels.
Runs are keyed by a content hash and appended to `nb_data/runs/…/runs.jsonl`, so rerunning this
cell resumes instead of repeating work.

In [ ]:
from ceilings.runner import load_runs, run_grid

t0 = time.time()
n_new = 0
for fam in FAMILIES:
    cells = experiment.main_cells(prepared[fam]["main"], prepared[fam]["table"], deltas=DELTAS)
    n_new += len(run_grid(cells, policy, OUT, save_transcripts=True))
records = load_runs(OUT, MODEL)
print(f"{n_new} new runs, {len(records)} total, {time.time() - t0:.1f}s")

## 5. Gain table and verdicts H1–H5

The gain is the paired difference in accuracy between an arm and baseline on the same instances,
with a cluster-bootstrap 95% interval. The prediction under test is the **diagonal**: E should help
only `info`, K only `reason`, and M only the `comm_*` families. A1 and the placebos should help
nowhere, and A2 should equal the single-agent reference with the same compute.

In [ ]:
from ceilings.analysis.gains import runs_frame
from ceilings.analysis.report import write_reports

df = runs_frame(records)
# One model here, so let a single evaluable model decide (the pre-registered rule needs 2).
res = write_reports(df[~df["is_sweep"]], OUT / "reports", delta="medium", n_boot=N_BOOT,
                    min_models=1)
gains = res["gain_table"]
pd.DataFrame([{"prediction": p.prediction_id, "verdict": p.verdict, "statement": p.statement}
              for p in res["predictions"] if p.prediction_id in {"H1", "H2", "H3", "H4", "H5"}])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

arm_order = ["E", "K", "M", "A1", "A2", "P1", "P2"]
med = gains[gains["delta"] == "medium"].pivot(index="family", columns="arm", values="gain")
med = med.reindex(index=FAMILIES, columns=arm_order)

fig, ax = plt.subplots(figsize=(8, 3.6))
im = ax.imshow(med.to_numpy(), cmap="RdBu", vmin=-1, vmax=1)
ax.set_xticks(range(len(arm_order)), arm_order)
ax.set_yticks(range(len(FAMILIES)), FAMILIES)
for i in range(med.shape[0]):
    for j in range(med.shape[1]):
        v = med.iat[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:+.2f}", ha="center", va="center", fontsize=9)
ax.set_title(f"Accuracy gain over baseline at medium delta ({MODEL})")
fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
fig.savefig(OUT / "reports" / "fig_gain_heatmap.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(FAMILIES), figsize=(14, 3.4), sharey=True)
for ax, fam in zip(axes, FAMILIES):
    g = gains[gains["family"] == fam]
    for arm in ["E", "K", "M", "A1", "P2"]:
        s = g[g["arm"] == arm].set_index("delta").reindex(list(DELTAS))
        ax.errorbar(s["added_tokens"], s["gain"],
                    yerr=[np.clip(s["gain"] - s["ci_low"], 0, None),
                          np.clip(s["ci_high"] - s["gain"], 0, None)],
                    marker="o", capsize=3, label=arm)
    ax.axhline(0, color="grey", lw=0.8)
    ax.set_title(fam)
    ax.set_xlabel("added tokens (realized)")
axes[0].set_ylabel("gain vs baseline")
axes[-1].legend(fontsize=8)
fig.suptitle("Gain against realized added cost, with 95% bootstrap intervals")
fig.tight_layout()
fig.savefig(OUT / "reports" / "fig_gain_vs_cost.png", dpi=150)
plt.show()

## 6. Fano floor sweep (H6)

The information family with k = 4 hidden bits (16 hosts), T probes at flip rate p. Corollary 2 /
Theorem 1 give an error floor from the budget T·(1 − h(p)), and the exact best-possible (MAP)
error is computed for T ≤ 10. Observed error must never fall significantly below the floor.

In [ ]:
cells_info, floors = experiment.info_sweep_cells(SWEEP_K, SWEEP_T, SWEEP_P, SIZES["n_sweep"],
                                                 root=OUT)
run_grid(cells_info, policy, OUT, save_transcripts=False)
answers = {c.instance.instance_id: c.instance for c in cells_info}

records = load_runs(OUT, MODEL)
df = runs_frame(records)
sw = df[df["is_sweep"] & (df["family"] == "info")]
err = sw.groupby(["flip_rate", "T"])["correct"].apply(lambda s: 1 - s.mean()).reset_index(name="error")

fig, ax = plt.subplots(figsize=(7, 4))
for p, g in err.groupby("flip_rate"):
    line, = ax.plot(g["T"], g["error"], "o-", label=f"observed, p={p}")
    Ts = sorted(SWEEP_T)
    ax.plot(Ts, [floors[(T, p)] for T in Ts], "--", color=line.get_color(), alpha=0.7,
            label=f"Fano floor, p={p}")
    exact_T = [T for T in Ts if T <= 10]
    ax.plot(exact_T, [theory.probe_map_error(SWEEP_K, p, T) for T in exact_T], ":",
            color=line.get_color(), label=f"exact MAP error, p={p}")
ax.set_xlabel("informative tool calls T")
ax.set_ylabel("error rate")
ax.set_title("Information family: error vs tool calls (k = 4)")
ax.legend(fontsize=7, ncol=2)
fig.tight_layout()
fig.savefig(OUT / "reports" / "fig_fano.png", dpi=150)
plt.show()

## 7. Communication threshold (H7)

Agent B may send agent A exactly L bits (bit mode). For parity, one bit is enough. For the inner
product, accuracy should stay near chance until a large share of the bits has crossed. The dashed
line is the highest accuracy that Theorem 5's bound allows (converted from bits with binary Fano):
no protocol may beat it. The simulated helper sends the first L bits of its list, a natural strategy
but not an optimal one.

In [ ]:
cells_comm = []
for variant in ("parity", "ip"):
    cells_comm += experiment.comm_sweep_cells(variant, SWEEP_NBITS, SWEEP_L, SIZES["n_sweep"],
                                              root=OUT)
run_grid(cells_comm, policy, OUT, save_transcripts=True)

records = load_runs(OUT, MODEL)
df = runs_frame(records)


def max_accuracy(h_theta, info_bits):
    # Binary Fano: h(Pe) >= H - I, so the best accuracy is 1 - h^{-1}(H - I).
    gap = max(0.0, h_theta - info_bits)
    lo, hi = 0.0, 0.5
    for _ in range(60):
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if theory.binary_entropy(mid) < gap else (lo, mid)
    return 1 - (lo + hi) / 2


h_ip = theory.theta_entropy_comm(SWEEP_NBITS, "ip")
fig, ax = plt.subplots(figsize=(7, 4))
for fam in ("comm_parity", "comm_ip"):
    g = df[df["is_sweep"] & (df["family"] == fam)].groupby("L")["correct"].mean()
    ax.plot(g.index, g.values, "o-", label=f"observed {fam}")
Ls = sorted(SWEEP_L)
ax.plot(Ls, [max_accuracy(h_ip, min(h_ip, theory.ip_bound(SWEEP_NBITS, L))) for L in Ls], "--",
        color="grey", label="Theorem 5 ceiling (inner product)")
ax.set_xlabel("message budget L (bits)")
ax.set_ylabel("accuracy")
ax.set_title(f"Communication family: accuracy vs bits sent (n = {SWEEP_NBITS})")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "reports" / "fig_comm_threshold.png", dpi=150)
plt.show()

## 8. Usable ≤ available ≤ budget, and the verdicts with sweeps

For the information sweep at p = 0.1:

- **usable information:** H(Θ) minus held-out log-loss of cross-fitted, temperature-calibrated
  predictions;
- **available information:** exact I(Θ; probe results), T ≤ 10;
- **budget:** T · (1 − h(p)).

The chain should hold up to estimation error.

In [ ]:
from ceilings.analysis.usable_info import usable_information

rows = []
sub = sw[sw["flip_rate"] == 0.1]
for T, g in sub.groupby("T"):
    insts = [answers[i] for i in g["instance_id"]]
    space = insts[0].answer_space
    probs = np.array([[pr[a] for a in space] for pr in g["probabilities"]])
    truth = np.array([space.index(i.answer) for i in insts])
    u = usable_information(probs, truth, float(SWEEP_K))
    rows.append({"T": int(T), "usable": u["usable_bits"], "usable_lo": u["ci_low"],
                 "usable_hi": u["ci_high"],
                 "available": theory.probe_information(SWEEP_K, 0.1, int(T)) if T <= 10 else np.nan,
                 "budget": min(SWEEP_K, T * theory.bsc_capacity(0.1))})
chain = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(7, 4))
ax.fill_between(chain["T"], chain["usable_lo"], chain["usable_hi"], alpha=0.2)
ax.plot(chain["T"], chain["usable"], "o-", label="usable (estimated)")
ax.plot(chain["T"], chain["available"], "s--", label="available (exact)")
ax.plot(chain["T"], chain["budget"], "k:", label="budget T·C")
ax.set_xlabel("tool calls T")
ax.set_ylabel("bits about the failing host")
ax.set_title("usable ≤ available ≤ budget (information family, p = 0.1)")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "reports" / "fig_chain.png", dpi=150)
plt.show()
chain.round(3)

In [ ]:
res = write_reports(df, OUT / "reports", delta="medium", floors=floors, n_boot=N_BOOT,
                    min_models=1)
verdicts = pd.DataFrame([{"prediction": p.prediction_id, "verdict": p.verdict,
                          "per_model": p.detail.get("per_model", ""), "statement": p.statement}
                         for p in res["predictions"]])
verdicts

## 9. Audit

The audit checks that:

- every record carries the pre-registration hash;
- token ledgers add up;
- bit-mode messages obeyed their budget;
- each arm's realized added cost stayed within ±5% of its target;
- refusal rates are ≤ 5%.

In [ ]:
import json

from ceilings import prereg
from ceilings.audit import audit

tables = {f"{MODEL}/{fam}": prepared[fam]["table"] for fam in FAMILIES}
report = audit(load_runs(OUT, MODEL), prereg.current_hash(), tables)
(OUT / "reports" / "audit.json").write_text(json.dumps(report, indent=2))
print("audit", "PASSED" if report["passed"] else "FAILED")
{k: (len(v), v[:3]) for k, v in report["checks"].items()}

## 10. Optional: tiny Claude pilot (off by default, costs money)

This runs baseline and the family's full intervention on a handful of `reason` instances with
`claude-opus-5-5`, stopping at `MAX_COST_USD`. It needs `ANTHROPIC_API_KEY` set in the
environment. It is a smoke test of the real adapter, not a calibrated pilot: the difficulty
(d = 12) was tuned for the simulated agent and is far too easy for a frontier model, so expect
baseline well above chance.

In [ ]:
RUN_CLAUDE = False
MAX_COST_USD = 2.00
N_CLAUDE = 5

if RUN_CLAUDE:
    from ceilings import arms, tasks
    from ceilings.runner import BudgetExceeded, Cell

    claude = make_policy("claude-opus-5-5")
    insts = tasks.generate_many("reason", N_CLAUDE, "calibration", d=12)
    cells = [Cell(i, arms.build("baseline", "reason")) for i in insts]
    cells += [Cell(i, arms.build("K", "reason", "high"), "medium") for i in insts]
    try:
        run_grid(cells, claude, OUT, max_cost=MAX_COST_USD, save_transcripts=True)
    except BudgetExceeded as e:
        print("stopped:", e)
    cdf = runs_frame(load_runs(OUT, "claude-opus-5-5"))
    print(cdf.groupby("arm")[["correct", "tokens", "dollars"]].mean())
    print("total spend: $", round(cdf["dollars"].sum(), 2))
else:
    print("Claude pilot skipped (RUN_CLAUDE = False)")

## 11. Save outputs

This bundles `nb_data/reports/` (the gain table, per-test results, prediction verdicts, audit and
figures) into one archive to download. Raw runs stay in `nb_data/runs/`.

In [ ]:
import shutil

archive = shutil.make_archive(str(OUT / "ceilings_reports"), "gztar", OUT / "reports")
print("download:", archive)
sorted(p.name for p in (OUT / "reports").iterdir())

## Reading the results

- **With `sim-bounded`, a clean diagonal is expected by construction.** It shows that the
  generators, cost matching, orchestrator and statistics recover a known structure. It says nothing
  about LLM teams.
- **"Inconclusive" on some controls at small scale is a power issue, not a failure.** It shows up
  for example on A1 in `comm_*`, where uninformed extra agents add voting noise. The corrected
  planning figure is about 850 paired instances per cell (research R10); `SCALE = "medium"` uses
  that.
- **The step to real evidence:** choose model families B and C (tasks T038/T039), recalibrate the
  difficulty of each family per model so baseline sits at chance, freeze the pre-registration
  (`ceilings prereg freeze`), then run the main grid.